In [ ]:
import sys
from itertools import product
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, IFrame, display

for candidate in (
    Path.cwd() / "Privacy-Scenario",
    Path.cwd(),
    Path.home() / "Desktop" / "COORDINATE-2.0" / "Privacy-Scenario",
):
    if (candidate / "run_privacy_experiments.py").exists():
        SCENARIO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Privacy-Scenario/run_privacy_experiments.py not found.")

sys.path.insert(0, str(SCENARIO_ROOT))
import run_privacy_experiments as scenario

PLOTS_DIR = SCENARIO_ROOT / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
summary_candidates = [
    scenario.RESULTS_ROOT / "privacy_summary.csv",
    scenario.RESULTS_ROOT / "smoke" / "privacy_summary.csv",
]
summary_path = next((path for path in summary_candidates if path.exists()), None)
if summary_path is None:
    raise FileNotFoundError("Run the privacy experiment campaign before generating plots.")

summary = pd.read_csv(summary_path)
required_columns = {
    "topology_label",
    "workload",
    "technique",
    "architecture_label",
    "avg_latency_s",
    "avg_throughput_tasks_per_hour",
    "avg_privacy_added_delay_seconds",
    "avg_adversary_success_rate",
    "avg_expected_distance_error",
}
missing_columns = sorted(required_columns - set(summary.columns))
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

topology_order = [name for name in scenario.TOPOLOGIES if name in set(summary["topology_label"])]
technique_order = [name for name in scenario.TECHNIQUES if name in set(summary["technique"])]
workload_order = [name for name in scenario.WORKLOADS if name in set(summary["workload"])]
arch_order = [name for name in scenario.ARCHITECTURES if name in set(summary["architecture_label"])]
technique_labels = {
    "none": "None",
    "encryption": "Encryption",
    "obfuscation": "Obfuscation",
    "combined": "Combined",
}
colors = {
    "FD": "#666666",
    "SD": "#999999",
    "CE": "#b3b3b3",
}

table_rows = []
for topology, workload, technique in product(topology_order, workload_order, technique_order):
    topology_idx = topology_order.index(topology)
    workload_idx = workload_order.index(workload)
    technique_idx = technique_order.index(technique)
    subset = summary[
        (summary["topology_label"] == topology)
        & (summary["workload"] == workload)
        & (summary["technique"] == technique)
    ]
    best_latency = subset["avg_latency_s"].min()
    best_throughput = subset["avg_throughput_tasks_per_hour"].max()
    topology_cell = topology.title() if workload_idx == len(workload_order) // 2 and technique_idx == len(technique_order) // 2 else ""
    workload_cell = workload.title() if technique_idx == len(technique_order) // 2 else ""
    latency_cells = []
    throughput_cells = []
    for arch in arch_order:
        row = subset[subset["architecture_label"] == arch]
        if row.empty:
            raise ValueError(f"Missing result for {topology}, {workload}, {technique}, {arch}.")
        latency = float(row["avg_latency_s"].iloc[0])
        throughput = float(row["avg_throughput_tasks_per_hour"].iloc[0])
        latency_text = f"{latency:.2f}"
        throughput_text = f"{throughput:.2f}"
        if latency == best_latency:
            latency_text = f"<strong>{latency_text}</strong>"
        if throughput == best_throughput:
            throughput_text = f"<strong>{throughput_text}</strong>"
        latency_cells.append(latency_text)
        throughput_cells.append(throughput_text)
    table_rows.append(
        "<tr>"
        f"<td>{topology_cell}</td><td>{workload_cell}</td><td>{technique_labels[technique]}</td>"
        + "".join(f"<td>{value}</td>" for value in latency_cells)
        + "".join(f"<td>{value}</td>" for value in throughput_cells)
        + "</tr>"
    )

table_html = """
<table style="border-collapse:collapse; font-size:12px; margin-bottom:16px; text-align:right;">
  <thead>
    <tr>
      <th rowspan="2" style="padding:4px 8px; text-align:left; border-bottom:1px solid #222;">Topology</th>
      <th rowspan="2" style="padding:4px 8px; text-align:left; border-bottom:1px solid #222;">Workload</th>
      <th rowspan="2" style="padding:4px 8px; text-align:left; border-bottom:1px solid #222;">Technique</th>
      <th colspan="3" style="padding:4px 8px; text-align:center; border-bottom:1px solid #222;">Latency (sec.)</th>
      <th colspan="3" style="padding:4px 8px; text-align:center; border-bottom:1px solid #222;">Throughput (tasks/hour)</th>
    </tr>
    <tr>
      <th style="padding:4px 8px;">CE</th><th style="padding:4px 8px;">SD</th><th style="padding:4px 8px;">FD</th>
      <th style="padding:4px 8px;">CE</th><th style="padding:4px 8px;">SD</th><th style="padding:4px 8px;">FD</th>
    </tr>
  </thead>
  <tbody>
""" + "\n".join(table_rows) + """
  </tbody>
</table>
"""
display(HTML(table_html))

plot_specs = [
    ("avg_latency_s", "Average Latency (sec.)", "latency_by_technique.pdf"),
    ("avg_throughput_tasks_per_hour", "Throughput (tasks/hour)", "throughput_by_technique.pdf"),
    ("avg_privacy_added_delay_seconds", "Privacy Overhead (sec.)", "privacy_overhead_by_technique.pdf"),
    ("avg_adversary_success_rate", "Adversary's Success Rate", "adversary_success_rate_by_technique.pdf"),
    ("avg_expected_distance_error", "Expected Distance Error (hops)", "expected_distance_error_by_technique.pdf"),
]

for plot_idx, (metric, ylabel, filename) in enumerate(plot_specs):
    fig, axes = plt.subplots(
        len(topology_order),
        len(workload_order),
        figsize=(3.0 * len(workload_order), 2.0 * len(topology_order)),
        sharey=False,
        squeeze=False,
    )

    for topology, workload in product(topology_order, workload_order):
        topology_idx = topology_order.index(topology)
        workload_idx = workload_order.index(workload)
        ax = axes[topology_idx][workload_idx]
        subset = summary[
            (summary["topology_label"] == topology)
            & (summary["workload"] == workload)
        ].copy()
        x = range(len(technique_order))
        width = 0.23
        ax.set_axisbelow(True)
        ax.grid(axis="y", linestyle=":", linewidth=0.5)

        for idx, arch in enumerate(arch_order):
            values = []
            for technique in technique_order:
                row = subset[
                    (subset["technique"] == technique)
                    & (subset["architecture_label"] == arch)
                ]
                if row.empty:
                    raise ValueError(f"Missing result for {topology}, {workload}, {technique}, {arch}.")
                values.append(pd.to_numeric(row[metric], errors="coerce").iloc[0])
            ax.bar(
                [i + (idx - 1) * width for i in x],
                values,
                width=width,
                label=arch,
                color=colors[arch],
                edgecolor="black",
                linewidth=0.4,
            )

        ax.set_title(f"{topology.title()} · {workload.title()} Workload")
        ax.set_xticks(list(x))
        ax.set_xticklabels([technique_labels[name] for name in technique_order], rotation=20, ha="right")
        ax.set_xlabel("Privacy Technique")
        ax.tick_params(axis="both", labelsize=8)
        if metric == "avg_adversary_success_rate":
            ax.set_ylim(0, 1.05)

        axes[topology_idx][0].set_ylabel(ylabel)
    if plot_idx == 0:
        handles, labels = axes[0][0].get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            title="Coordination Mechanisms",
            loc="lower center",
            bbox_to_anchor=(0.5, 1.02),
            ncol=len(arch_order),
            frameon=False,
        )

    fig.tight_layout()
    pdf_path = PLOTS_DIR / filename
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    display(IFrame(src=f"plots/{filename}", width="100%", height=320))
